# Daily Vehicle Data Pipeline (data.gov.il)

This notebook implements the **daily ingestion, cleaning, and comparison pipeline**
for the Israeli vehicle registry dataset (data.gov.il).

## High-level flow
1. Download the full dataset daily using API paging
2. Maintain a rolling snapshot structure (latest + yesterday)
3. Clean and standardize the data
4. Compare yesterday vs today to detect added/removed vehicles
5. Allow ad-hoc lookup of a specific vehicle by plate number

The notebook is executed daily via a Databricks Job.


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.gov_data;


In [0]:
import datetime
import requests
import pandas as pd
from pyspark.sql import functions as F

BASE_DIR_VOL = "dbfs:/Volumes/workspace/default/gov_data"

API_BASE_URL = "https://data.gov.il/api/3/action/datastore_search"
RESOURCE_ID  = "053cea08-09bc-40ec-8f7a-156f0677aff3"
PAGE_SIZE    = 100_000

PLATE_COLUMN = "mispar_rechev"

def vol_path(*parts: str) -> str:
    return "/".join([BASE_DIR_VOL.rstrip("/")] + [p.strip("/") for p in parts])

def ensure_dir(path: str):
    try:
        dbutils.fs.ls(path)
    except Exception:
        dbutils.fs.mkdirs(path)

# folder structure
RAW_DIR     = vol_path("raw")
CLEAN_DIR   = vol_path("clean")
REPORTS_DIR = vol_path("reports")

for p in [RAW_DIR, CLEAN_DIR, REPORTS_DIR]:
    ensure_dir(p)

print("Using volume:", BASE_DIR_VOL)
print("RAW:", RAW_DIR)
print("CLEAN:", CLEAN_DIR)
print("REPORTS:", REPORTS_DIR)


## Task 1 – Daily Ingestion & Snapshot Management

This step downloads the full vehicle dataset from data.gov.il
and maintains a rolling snapshot strategy:

- `vehicles_latest.csv` – current daily snapshot
- `vehicles_<YYYY-MM-DD>.csv` – previous day's snapshot

The task is executed daily via a Databricks Job.


In [0]:
def rotate_yesterday_file(today: datetime.date | None = None) -> None:
    if today is None:
        today = datetime.date.today()

    y = today - datetime.timedelta(days=1)
    y_str = y.strftime("%Y-%m-%d")

    latest = vol_path("raw", "vehicles_latest.csv")
    yfile  = vol_path("raw", f"vehicles_{y_str}.csv")

    # אם אין latest עדיין
    try:
        dbutils.fs.ls(latest)
    except Exception:
        print("No latest file to rotate.")
        return

    # אם אתמול כבר קיים, לא עושים כלום
    try:
        dbutils.fs.ls(yfile)
        print("Yesterday file already exists:", yfile)
        return
    except Exception:
        pass

    dbutils.fs.mv(latest, yfile)
    print("Rotated latest →", yfile)


def download_full_to_latest() -> str:
    """
    Download full dataset using paging.
    Write each page to tmp_in WITHOUT header.
    Then read tmp_in as header=false, assign real column names, and write ONE final CSV with header.
    """
    run_id = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    tmp_in  = vol_path(f"_tmp_in_{run_id}")     # pages appended here (folders/parts)
    tmp_out = vol_path(f"_tmp_out_{run_id}")    # final coalesce(1) folder

    # cleanup
    for p in [tmp_in, tmp_out]:
        try:
            dbutils.fs.rm(p, recurse=True)
        except Exception:
            pass

    offset = 0
    total = 0
    cols = None  # will capture real column order from first page

    print("Downloading pages... writing to:", tmp_in)

    while True:
        params = {"resource_id": RESOURCE_ID, "limit": PAGE_SIZE, "offset": offset}
        resp = requests.get(API_BASE_URL, params=params, timeout=60)
        resp.raise_for_status()
        records = resp.json()["result"]["records"]

        if not records:
            print("No more records. Finished.")
            break

        pdf = pd.DataFrame(records)

        # capture real column names once
        if cols is None:
            cols = list(pdf.columns)

        # Spark dataframe as strings (safe)
        sdf = spark.createDataFrame(pdf.astype(str))

        # Append WITHOUT header always (prevents mixed header parts)
        (sdf.write.mode("append")
            .option("header", "false")
            .option("escape", '"')
            .option("quote", '"')
            .option("encoding", "UTF-8")
            .csv(tmp_in))

        offset += len(records)
        total  += len(records)

        print(f"Page +{len(records):,} | total={total:,} | offset={offset:,}")

        if len(records) < PAGE_SIZE:
            print("Last page reached.")
            break

    if total == 0:
        raise RuntimeError("No data downloaded from API.")

    if not cols:
        raise RuntimeError("Could not infer columns from first page.")

    # Read tmp_in with NO header -> columns will be _c0,_c1,...
    df_all = spark.read.option("header", "false").csv(tmp_in)

    # Assign real column names
    df_all = df_all.toDF(*cols)

    # Write ONE final CSV with header
    (df_all.coalesce(1)
          .write.mode("overwrite")
          .option("header", "true")
          .csv(tmp_out))

    # move single part to vehicles_latest.csv
    part_files = [f.path for f in dbutils.fs.ls(tmp_out)
                  if f.name.startswith("part-") and f.name.endswith(".csv")]
    if not part_files:
        raise RuntimeError("No part file produced in tmp_out.")

    latest = vol_path("raw", "vehicles_latest.csv")

    try:
        dbutils.fs.rm(latest)
    except Exception:
        pass

    dbutils.fs.mv(part_files[0], latest)

    # cleanup temp folders
    dbutils.fs.rm(tmp_in,  recurse=True)
    dbutils.fs.rm(tmp_out, recurse=True)

    print("Saved latest snapshot to Volume:", latest)
    return latest


def run_daily_task_1():
    rotate_yesterday_file()
    return download_full_to_latest()


# מה שה-Job היומי יריץ:
latest_path = run_daily_task_1()
latest_path


## Data Cleaning & Standardization

This step ensures data quality and consistency by:
- Removing duplicate vehicles based on plate number
- Normalizing ownership type
- Converting manufacturing year to a numeric format

All downstream processing relies on the clean dataset.


In [0]:
PLATE_COLUMN = "mispar_rechev"

RAW_DIR   = vol_path("raw")
CLEAN_DIR = vol_path("clean")

ensure_dir(RAW_DIR)
ensure_dir(CLEAN_DIR)

def clean_latest_snapshot(today=None) -> str:
    if today is None:
        today = datetime.date.today()
    d = today.strftime("%Y-%m-%d")

    raw_latest   = vol_path("raw", "vehicles_latest.csv")
    clean_latest = vol_path("clean", "vehicles_latest_clean.csv")

    df = spark.read.option("header", "true").csv(raw_latest)

    # --- Cleansing / standardization ---
    if PLATE_COLUMN in df.columns:
        df = df.withColumn(PLATE_COLUMN, F.trim(F.col(PLATE_COLUMN).cast("string")))
        df = df.filter(F.col(PLATE_COLUMN).isNotNull() & (F.col(PLATE_COLUMN) != ""))
        df = df.dropDuplicates([PLATE_COLUMN])

    # shnat_yitzur -> int + basic sanity range
    if "shnat_yitzur" in df.columns:
        df = df.withColumn("shnat_yitzur_int", F.col("shnat_yitzur").cast("int"))
        df = df.filter((F.col("shnat_yitzur_int").isNull()) | ((F.col("shnat_yitzur_int") >= 1950) & (F.col("shnat_yitzur_int") <= datetime.date.today().year)))

    # optional: normalize ownership to 2 buckets (private/commercial-ish)
    if "baalut" in df.columns:
        df = df.withColumn("baalut_norm", F.when(F.col("baalut").contains("פרטי"), F.lit("private")).otherwise(F.lit("other")))

    # write clean latest as single csv
    tmp_out = vol_path("clean", f"_tmp_clean_{d}")
    try: dbutils.fs.rm(tmp_out, recurse=True)
    except: pass

    (df.coalesce(1)
       .write.mode("overwrite")
       .option("header", "true")
       .csv(tmp_out))

    part = [f.path for f in dbutils.fs.ls(tmp_out) if f.name.startswith("part-") and f.name.endswith(".csv")][0]
    try: dbutils.fs.rm(clean_latest)
    except: pass

    dbutils.fs.mv(part, clean_latest)
    dbutils.fs.rm(tmp_out, recurse=True)
    
    print("Saved clean latest:", clean_latest)

    # also save a dated clean snapshot (needed for Task 2 "yesterday vs latest")
    clean_daily = vol_path("clean", f"vehicles_{d}_clean.csv")
    try:
        dbutils.fs.rm(clean_daily)
    except Exception:
        pass
    dbutils.fs.cp(clean_latest, clean_daily)
    print("Also saved clean daily:", clean_daily)

    return clean_latest

clean_latest_snapshot()

## Task 2 – Daily Change Detection

This task compares yesterday’s clean snapshot with today’s
latest clean snapshot in order to identify:

- Newly added vehicles
- Removed vehicles

Comparison is based on the vehicle plate number (business key).


In [0]:
def compare_two_paths(label1: str, path1: str, label2: str, path2: str) -> str:
    df1 = spark.read.option("header", "true").csv(x)
    df2 = spark.read.option("header", "true").csv(path2)

    # keys only (dedup)
    k1 = df1.select(PLATE_COLUMN).dropDuplicates()
    k2 = df2.select(PLATE_COLUMN).dropDuplicates()

    # ADDED keys: in day2 not in day1
    added_keys = k2.join(k1, on=PLATE_COLUMN, how="left_anti")

    # REMOVED keys: in day1 not in day2
    removed_keys = k1.join(k2, on=PLATE_COLUMN, how="left_anti")

    # Bring full rows (details)
    added = (df2.join(added_keys, on=PLATE_COLUMN, how="inner")
               .withColumn("change_type", F.lit("ADDED"))
               .withColumn("source", F.lit(label2)))

    removed = (df1.join(removed_keys, on=PLATE_COLUMN, how="inner")
                .withColumn("change_type", F.lit("REMOVED"))
                .withColumn("source", F.lit(label1)))

    report = added.unionByName(removed, allowMissingColumns=True)

    out_name = f"diff_report_{label1}_vs_{label2}.csv"
    out_path = vol_path("reports", out_name)
    tmp_out  = vol_path("reports", f"_tmp_{out_name}")  

    try: dbutils.fs.rm(tmp_out, recurse=True)
    except: pass

    (report.coalesce(1)
           .write.mode("overwrite")
           .option("header", "true")
           .csv(tmp_out))

    part_files = [f.path for f in dbutils.fs.ls(tmp_out)
                  if f.name.startswith("part-") and f.name.endswith(".csv")]
    if not part_files:
        raise RuntimeError("No diff part file produced.")

    try: dbutils.fs.rm(out_path)
    except: pass

    dbutils.fs.mv(part_files[0], out_path)
    dbutils.fs.rm(tmp_out, recurse=True)

    print("Diff report saved:", out_path)
    return out_path


def run_daily_task_2():
    today = datetime.date.today()
    y = today - datetime.timedelta(days=1)

    label_y = y.strftime("%Y-%m-%d")
    label_t = today.strftime("%Y-%m-%d") + "_latest"

    p_yesterday = vol_path("clean", f"vehicles_{label_y}_clean.csv")
    p_latest = vol_path("clean", "vehicles_latest_clean.csv")

    try:
        dbutils.fs.ls(p_yesterday)
        dbutils.fs.ls(p_latest)
    except Exception:
        print("Missing files for diff. Need vehicles_<yesterday>.csv and vehicles_latest.csv. Skipping Task 2.")
        return None

    return compare_two_paths(label_y, p_yesterday, label_t, p_latest)


diff_path = run_daily_task_2()
diff_path


## Task 3 – Vehicle Lookup by Plate Number

This utility task allows querying the latest cleaned dataset
for a specific vehicle using its registration number.

**Purpose:**
- Enable quick validation of a specific vehicle record
- Support ad-hoc analysis and investigation

**Output:**
- A single CSV file containing the vehicle record (if found)
- The file is saved under the `reports/` directory

This task is intended for manual execution (not scheduled).


In [0]:
def search_vehicle_by_plate_latest(plate: str) -> str | None:
    p = vol_path("clean", "vehicles_latest_clean.csv")
    df = spark.read.option("header", "true").csv(p)

    res = df.filter(F.col(PLATE_COLUMN) == F.lit(str(plate)))

    if res.limit(1).count() == 0:
        print(f"No record found for plate {plate} in latest clean")
        return None

    out_name = f"plate_{plate}_latest.csv"
    tmp_out  = vol_path("reports", f"_tmp_plate_{plate}_latest")
    out_path = vol_path("reports", out_name)

    try: dbutils.fs.rm(tmp_out, recurse=True)
    except: pass

    (res.coalesce(1)
        .write.mode("overwrite")
        .option("header", "true")
        .csv(tmp_out))

    part_files = [f.path for f in dbutils.fs.ls(tmp_out) if f.name.startswith("part-") and f.name.endswith(".csv")]
    if not part_files:
        raise RuntimeError("No plate part file produced.")

    try: dbutils.fs.rm(out_path)
    except: pass

    dbutils.fs.mv(part_files[0], out_path)
    dbutils.fs.rm(tmp_out, recurse=True)

    print("Saved plate record:", out_path)
    return out_path

search_vehicle_by_plate_latest("CAR_NUMBER")



In [0]:
def create_presentation_tables(diff_csv_path: str, plate_csv_path: str, n_head: int = 100):
    # 1 DIFF TABLE
    diff_df = spark.read.option("header", "true").csv(diff_csv_path)
    diff_df.write.mode("overwrite").saveAsTable("workspace.default.diff_report_latest")

    # 2 PLATE SEARCH TABLE
    plate_df = spark.read.option("header", "true").csv(plate_csv_path)
    plate_df.write.mode("overwrite").saveAsTable("workspace.default.plate_search_latest")

    # 3 CLEAN - first N rows
    clean_path = vol_path("clean", "vehicles_latest_clean.csv")
    clean_head = spark.read.option("header", "true").csv(clean_path).limit(n_head)
    clean_head.write.mode("overwrite").saveAsTable(f"workspace.default.vehicles_clean_head_{n_head}")

    # 4 RAW - first N rows
    raw_path = vol_path("raw", "vehicles_latest.csv")
    raw_head = (spark.read.option("header", "true").csv(raw_path).limit(n_head))
    raw_head.write.mode("overwrite").saveAsTable(f"workspace.default.vehicles_raw_head_{n_head}")

    print("Presentaion tables created:")
    print(" - diff_report_latest")
    print(" - plate_search_latest")
    print(f" - vehicles_clean_head_{n_head}")
    print(f" - vehicles_raw_head_{n_head}")


# RUN ON PRESENTATION:
diff_csv_path = vol_path("reports", "diff_report_DATE_1_vs_DATE_2_latest.csv")
plate_csv_path = vol_path("reports", "plate_CAR_NUMBER_latest.csv")

create_presentation_tables(diff_csv_path, plate_csv_path)








